# 作業四：LLM Function Calling

本作業要實作一個 **Function Calling AI Agent**，核心是下面的兩輪流程：

```
使用者提問
  → Round 1：LLM 判斷要不要呼叫工具，輸出 JSON
      • 需要工具：{"tool": "...", "arguments": {...}}
      • 不需要工具（閒聊）：{"final": "..."}
  → Python 路由：依工具名查 mock_data.json
  → Round 2：把查到的資料丟回 LLM，要它輸出自然語言 {"final": "..."}
```

你需要練習的不是寫聊天機器人，而是：

1. 用 **system prompt** 約束模型「該用工具就用、不該用就閒聊」並只吐 JSON。
2. 設計 **tool schema**，讓模型在工具中選對的、抽對的參數。
3. 在 Python 中寫 **handle_tool_call**，把模型輸出路由到對應的查詢函式。
4. 設計一個你自己的 **custom tool**，從現有資料推導新結果。

## 環境準備

1. 到 [console.groq.com](https://console.groq.com) 免費申請 API Key
2. `cp .env.example .env`，把 key 貼進 `.env` 的 `GROQ_API_KEY=`

## 評分
- 共 110 分
- Q1–Q5 共 50 分（每題 10 分）
- 自訂工具 10 分
- 報告 40 分
- 加分題：使用兩個以上模型比較 +10

## 0. 載入工具與資料庫

`fc_utils.py` 是助教提供的共用工具（你不需要修改它），裡面有：

- `call_llm(messages)`：呼叫 Groq Qwen3-32B 或其他模型
- `parse_json_output(text)`：從模型輸出抓 JSON
- `lookup_db(section, key)`：查 mock_data 的工具函式
- `load_db(path)`：載入 mock_data.json
- `run_tests(run_agent, db)`：跑 5 題並輸出 `fc_log.json`

In [4]:
!pip install groq

In [5]:
import sys
import importlib.metadata

def get_ver(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return "Not Installed"

# 針對 nlp_as4.ipynb 與 fc_utils.py 實際用到的項目
package_info = [
    ("Python", sys.version.split()[0]),
    ("groq", get_ver("groq")),
    ("python-dotenv", get_ver("python-dotenv")),
    ("json (Std Lib)", sys.version.split()[0]),
    ("re (Std Lib)", sys.version.split()[0]),
    ("os (Std Lib)", sys.version.split()[0]),
    ("pathlib (Std Lib)", sys.version.split()[0]),
]

def print_version_table(data):
    width = 50
    print("=" * width)
    print(f"{'項目 (Package)':<26} | {'版本 (Version)':<17}")
    print("-" * 45)

    for pkg, ver in data:
        print(f"{pkg:<26} | {ver:<17}")

    print("=" * width)

print_version_table(package_info)

項目 (Package)               | 版本 (Version)     
---------------------------------------------
Python                     | 3.12.13          
groq                       | 1.2.0            
python-dotenv              | 1.2.2            
json (Std Lib)             | 3.12.13          
re (Std Lib)               | 3.12.13          
os (Std Lib)               | 3.12.13          
pathlib (Std Lib)          | 3.12.13          


In [6]:
import json
from fc_utils import (
    call_llm,
    parse_json_output,
    lookup_db,
    load_db,
    run_tests,
    set_model,
)

db = load_db("mock_data.json")
print("資料庫類別：", list(db.keys()))
print("範例（AAPL）：", db["stocks"]["data"]["AAPL"])

資料庫類別： ['meta', 'stocks', 'weather', 'news']
範例（AAPL）： {'price': 150.25, 'currency': 'USD', 'name': 'Apple Inc.', 'change_pct': 1.2}


In [7]:
# TODO1: 指定模型名稱
# 加分題（+10）：在報告中比較至少兩個模型的結果。
# 請分別跑 run_tests，把產出檔重新命名為 fc_log_<模型名>.json 一併繳交，並在報告中分析差異。

set_model("qwen/qwen3-32b")  # https://console.groq.com/docs/models
#set_model("llama-3.3-70b-versatile")

## 1. Round 1 System Prompt

Round 1 的目標：讓模型看到使用者問題後，**只**輸出一個 JSON。可能是：

- 需要呼叫工具：`{"tool": "...", "arguments": {...}}`
- 閒聊或與工具無關：`{"final": "<繁體中文回答>"}`

好的 prompt 通常會：

1. 列出所有可用工具與其參數格式
2. 用範例示範兩種 JSON 格式
3. 強制不准輸出 JSON 以外的東西
4. 強制不准猜測股價、天氣（必須靠工具）

In [8]:
# TODO2: 請修改 SYSTEM_PROMPT_R1，目前已經提供最基礎的 prompt
# 請完善它使模型能順利進行 function calling

SYSTEM_PROMPT_R1 = """你是專精於「決策路由」的 Function Calling AI 助理。
你的任務是解析使用者提問，並嚴格根據以下規則輸出一個且僅一個合法的 JSON 物件。

可用工具：
- get_stock_price：查詢股票即時價格 → 參數 {"symbol": "AAPL"}
- get_weather：查詢城市即時天氣 → 參數 {"city": "Taipei"}
- get_news：查詢分類新聞 → 參數 {"category": "tech"}（可選 tech / finance / sports）
- get_stock_price_diff：計算兩支股票的絕對價差 → 參數 {"symbol_a": "第一支股票代碼", "symbol_b": "第二支股票代碼"}

輸出格式：
情況 A：需要呼叫工具（涉及即時數據、股價、天氣、新聞）。
   - 格式：{"tool": "工具名稱", "arguments": {參數鍵值對}}
情況 B：不需要工具（如閒聊、自我介紹、解釋概念、參數不明）。
   - 格式：{"final": "繁體中文自然語言回答"}

【強制規則】
1. 只輸出 JSON，不要有任何 Markdown 標記或解釋文字。
2. 涉及股價、天氣、新聞等即時數據，必須呼叫工具，不得自行編造。
3. 若參數缺失（如問「這支股票多少錢」），請透過 final 詢問具體代碼。
4. 回答必須使用繁體中文。

範例
- 輸入：「AAPL 現在多少錢？」
  輸出：{"tool": "get_stock_price", "arguments": {"symbol": "AAPL"}}
- 輸入：「你好，你是誰？」
  輸出：{"final": "你好！我是你的 AI 助理，可以幫你查詢股價、天氣以及最新新聞。"}
- 輸入：「我想看 2330 和輝達的價差。」
  輸出：{"tool": "get_stock_price_diff", "arguments": {"symbol_a": "2330", "symbol_b": "NVDA"}}

"""
print(SYSTEM_PROMPT_R1)

你是專精於「決策路由」的 Function Calling AI 助理。
你的任務是解析使用者提問，並嚴格根據以下規則輸出一個且僅一個合法的 JSON 物件。

可用工具：
- get_stock_price：查詢股票即時價格 → 參數 {"symbol": "AAPL"}
- get_weather：查詢城市即時天氣 → 參數 {"city": "Taipei"}
- get_news：查詢分類新聞 → 參數 {"category": "tech"}（可選 tech / finance / sports）
- get_stock_price_diff：計算兩支股票的絕對價差 → 參數 {"symbol_a": "第一支股票代碼", "symbol_b": "第二支股票代碼"}

輸出格式：
情況 A：需要呼叫工具（涉及即時數據、股價、天氣、新聞）。
   - 格式：{"tool": "工具名稱", "arguments": {參數鍵值對}}
情況 B：不需要工具（如閒聊、自我介紹、解釋概念、參數不明）。
   - 格式：{"final": "繁體中文自然語言回答"}

【強制規則】
1. 只輸出 JSON，不要有任何 Markdown 標記或解釋文字。
2. 涉及股價、天氣、新聞等即時數據，必須呼叫工具，不得自行編造。
3. 若參數缺失（如問「這支股票多少錢」），請透過 final 詢問具體代碼。
4. 回答必須使用繁體中文。

範例
- 輸入：「AAPL 現在多少錢？」
  輸出：{"tool": "get_stock_price", "arguments": {"symbol": "AAPL"}}
- 輸入：「你好，你是誰？」
  輸出：{"final": "你好！我是你的 AI 助理，可以幫你查詢股價、天氣以及最新新聞。"}
- 輸入：「我想看 2330 和輝達的價差。」
  輸出：{"tool": "get_stock_price_diff", "arguments": {"symbol_a": "2330", "symbol_b": "NVDA"}}




## 2. Round 2 System Prompt

Round 2 的目標：把 Python 查到的資料餵給模型，讓它生成自然語言回答。

`build_round2_prompt` 是一個會根據工具結果動態組裝 prompt 的函式。重點：

- 工具回傳 `ok=true` 時，必須使用 `data` 中的具體數值回答。
- 工具回傳 `ok=false` 時，必須回「查無該資料」，不能憑空編造。
- 仍然只輸出 JSON `{"final": "..."}`。

In [9]:
def build_round2_prompt(function_name: str, result_json: str) -> str:
    return f"""工具「{function_name}」查詢結果如下：
{result_json}

請根據以上資料用繁體中文回答使用者的問題。輸出格式：
{{"final": "<自然語言回答>"}}

【規則】
- 當 ok=true：必須引用 data 中的具體數值或內容（例如：價格、溫度、新聞標題），不得另外編造資訊。
- 當 ok=false：直接輸出 {{"final": "查無該資料。"}}。
- 只輸出 JSON，不要有任何其他文字。
"""

print(build_round2_prompt("get_stock_price", '{"ok": true, "data": {"price": 150.25, "currency": "USD"}}'))

工具「get_stock_price」查詢結果如下：
{"ok": true, "data": {"price": 150.25, "currency": "USD"}}

請根據以上資料用繁體中文回答使用者的問題。輸出格式：
{"final": "<自然語言回答>"}

【規則】
- 當 ok=true：必須引用 data 中的具體數值或內容（例如：價格、溫度、新聞標題），不得另外編造資訊。
- 當 ok=false：直接輸出 {"final": "查無該資料。"}。
- 只輸出 JSON，不要有任何其他文字。



## 3. Tool Schema

Tool schema 是給模型看的「工具說明書」。模型會根據 `description` 和 `parameters` 來判斷：

- 我該用哪個工具？
- 我該抽哪些參數？

好的 description 會具體寫出**使用情境**和**範例參數**，而不是只寫工具名稱。

底下定義 3 個基礎工具，你需要再做 1 個自訂工具（TODO4）。

In [10]:
# TODO3: 請填寫各個 tool 的 description、parameters 內的 `type` 和 `required`
# Hint: 可參考 `get_stock_avg_price`

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_stock_price",
            "description": "TODO3",
            "parameters": {
                "type": "object",
                "properties": {
                    "symbol": {
                        "type": "TODO3",
                        "description": "股票代碼，例如 'AAPL'、'TSLA'、'TSMC'，或中文別名如『蘋果』、『特斯拉』、『台積電』",
                    }
                },
                "required": ["TODO3"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "TODO3",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "TODO3",
                        "description": "城市英文名，例如 'Taipei'、'Tokyo'、'London'，或中文別名如『台北』、『東京』、『倫敦』",
                    }
                },
                "required": ["TODO3"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_news",
            "description": "TODO3",
            "parameters": {
                "type": "object",
                "properties": {
                    "category": {
                        "type": "TODO3",
                        "description": "新聞類別：'tech'（科技）、'finance'（財經）、'sports'（運動）",
                        "enum": ["tech", "finance", "sports"],
                    }
                },
                "required": ["TODO3"],
            },
        },
    },
    # TODO4: 實作自訂工具
    # 範例：計算兩支股票的平均股價
    # {
    #     "type": "function",
    #     "function": {
    #         "name": "get_stock_avg_price",
    #         "description": "計算兩支股票的平均股價（兩支股票必須使用相同貨幣）。當使用者要比較或求兩股票均價時使用。",
    #         "parameters": {
    #             "type": "object",
    #             "properties": {
    #                 "symbol_a": {"type": "string", "description": "第一支股票代碼"},
    #                 "symbol_b": {"type": "string", "description": "第二支股票代碼"},
    #             },
    #             "required": ["symbol_a", "symbol_b"],
    #         },
    #     },
    # },

    # TODO4: 兩支股票的價差計算
    {
        "type": "function",
        "function": {
            "name": "get_stock_price_diff",
            "description": "計算兩支股票之間的絕對價差。僅限於相同幣別的股票比較。",
            "parameters": {
                "type": "object",
                "properties": {
                    "symbol_a": {"type": "string", "description": "第一支股票代碼"},
                    "symbol_b": {"type": "string", "description": "第二支股票代碼"},
                },
                "required": ["symbol_a", "symbol_b"],
            },
        },
    }
]


print(f"共定義 {len(TOOLS)} 個工具：", [t["function"]["name"] for t in TOOLS])

共定義 4 個工具： ['get_stock_price', 'get_weather', 'get_news', 'get_stock_price_diff']


## 4. handle_tool_call：路由與資料庫查詢

Round 1 的模型輸出告訴我們「要呼叫哪個工具、用什麼參數」，但**真正的查詢動作要在 Python 端做**。`handle_tool_call` 負責這個：

- 根據 `function_name` 決定走哪個分支
- 用 `lookup_db` 從 `mock_data.json` 撈資料
- 統一回傳格式：`{"ok": true, "data": ...}` 或 `{"ok": false, "error": "not_found"}`

**重要**：自訂工具的結果必須**從 mock_data 推導**，不能 hardcode 答案。

In [11]:
def handle_tool_call(function_name: str, function_args: dict, db: dict) -> str:
    result = None

    if function_name == "get_stock_price":
        symbol = function_args.get("symbol", "")
        result = lookup_db(db.get("stocks", {}), symbol)

    elif function_name == "get_weather":
        city = function_args.get("city", "")
        result = lookup_db(db.get("weather", {}), city)

    elif function_name == "get_news":
        category = function_args.get("category", "")
        result = lookup_db(db.get("news", {}), category)

    # --- TODO4: 股價價差 ---
    elif function_name == "get_stock_price_diff":
        # 從參數中取得兩個股票代碼
        symbol_a = function_args.get("symbol_a", "")
        symbol_b = function_args.get("symbol_b", "")

        # 從資料庫查找資料
        a = lookup_db(db.get("stocks", {}), symbol_a)
        b = lookup_db(db.get("stocks", {}), symbol_b)

        # 兩者都必須存在且貨幣單位相同
        if a and b and a.get("currency") == b.get("currency"):
            diff = abs(a["price"] - b["price"])
            result = {
                "symbol_a": symbol_a,
                "symbol_b": symbol_b,
                "price_a": a["price"],
                "price_b": b["price"],
                "difference": round(diff, 2),
                "currency": a["currency"]
            }

    else:
        return json.dumps({"ok": False, "error": "unknown_tool"}, ensure_ascii=False)

    if result is not None:
        return json.dumps({"ok": True, "data": result}, ensure_ascii=False)
    return json.dumps({"ok": False, "error": "not_found"}, ensure_ascii=False)

# 測試
print(handle_tool_call("get_stock_price_diff", {"symbol_a": "AAPL", "symbol_b": "NVDA"}, db))

{"ok": true, "data": {"symbol_a": "AAPL", "symbol_b": "NVDA", "price_a": 150.25, "price_b": 875.4, "difference": 725.15, "currency": "USD"}}


## 5. run_agent：兩輪 Function Calling 主流程

把上面四件事串起來。流程：

1. **Round 1**：用 `SYSTEM_PROMPT_R1` 問模型，解析 JSON。
   - 如果是 `{"final": ...}` → 直接回答（閒聊路徑）
   - 如果是 `{"tool": ..., "arguments": ...}` → 進入 Round 2
2. **工具呼叫**：用 `handle_tool_call` 查 mock_data。
3. **Round 2**：用 `build_round2_prompt(工具名, 工具結果)` 問模型，要它寫成自然語言。

回傳 `{"model_call": ..., "final_answer": ...}`，這就是寫進 `fc_log.json` 的內容。

In [12]:
def run_agent(query: str, db: dict) -> dict:
    # ── Round 1：判斷是否使用工具 ──
    r1_messages = [
        {"role": "system", "content": SYSTEM_PROMPT_R1},
        {"role": "user", "content": query},
    ]
    r1_raw = call_llm(r1_messages, max_tokens=1024)
    r1_json = parse_json_output(r1_raw)

    # 閒聊路徑
    if "final" in r1_json and "tool" not in r1_json:
        return {"model_call": "無呼叫工具", "final_answer": r1_json["final"]}

    function_name = r1_json.get("tool", "")
    function_args = r1_json.get("arguments", {})
    if not function_name:
        return {"model_call": "無呼叫工具（格式錯誤）", "final_answer": r1_raw}

    model_call_info = f"{function_name}({json.dumps(function_args, ensure_ascii=False)})"

    # ── 工具呼叫（Python 端查 mock_data）──
    tool_result = handle_tool_call(function_name, function_args, db)

    # ── Round 2：用工具結果生成自然語言 ──
    r2_messages = [
        {"role": "system", "content": build_round2_prompt(function_name, tool_result)},
        {"role": "user", "content": query},
    ]
    r2_raw = call_llm(r2_messages, max_tokens=2048)
    r2_json = parse_json_output(r2_raw)
    final_answer = r2_json.get("final", r2_raw)

    return {"model_call": model_call_info, "final_answer": final_answer}

## 6. 單題除錯

正式跑 5 題前，建議先用單題確認 prompt 與 schema 是否合理。

In [13]:
# GROQ_API_KEY
import os
os.environ["GROQ_API_KEY"] = "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"

In [14]:
result = run_agent("台北的天氣如何？", db)
print("model_call :", result["model_call"])
print("final_answer:", result["final_answer"])

model_call : get_weather({"city": "Taipei"})
final_answer: 台北目前氣溫25°C，天氣多雲，濕度70%，風速12公里/小時。


In [15]:
# TODO4: 完成 TODO4 前兩段後，請寫一個能觸發你的自訂工具的 query 來驗證
# 範例（示範格式，AAPL+TSLA 是針對註解中的 get_stock_avg_price）：
# result = run_agent("請問 AAPL 跟 TSLA 兩支股票的平均股價？", db)
# print("model_call :", result["model_call"])
# print("final_answer:", result["final_answer"])

# 你的測試（請改成觸發你自訂工具的 query）：
# result = run_agent("<你的 query>", db)
# print("model_call :", result["model_call"])
# print("final_answer:", result["final_answer"])
# TODO4: 完成 TODO4 前兩段後，請寫一個能觸發你的自訂工具的 query 來驗證

query = "請問蘋果 (AAPL) 和輝達 (NVDA) 的股價差多少？"
result = run_agent(query, db)
print("model_call :", result["model_call"])
print("final_answer:", result["final_answer"])

model_call : get_stock_price_diff({"symbol_a": "AAPL", "symbol_b": "NVDA"})
final_answer: 蘋果 (AAPL) 股價為 150.25 美元，輝達 (NVDA) 股價為 875.4 美元，兩者股價差為 725.15 美元。


## 7. 正式測驗（5 題，輸出 fc_log.json）

`run_tests` 會跑下列 5 題並把結果寫到 `fc_log.json`，這是你最終要繳交的檔案：

| Q | 題目 | 測什麼 |
|---|---|---|
| Q1 | AAPL 股價 | 工具選擇與參數抽取 |
| Q2 | Taipei 天氣 | 工具選擇與參數抽取 |
| Q3 | tech 新聞 | 工具選擇與參數抽取 |
| Q4 | MSFT 股價（資料庫沒有）| 抗幻覺：能否回「查無資料」 |
| Q5 | 「你是什麼人工智慧模型？」 | 能否判斷不該呼叫工具 |

In [16]:
results = run_tests(run_agent, db)

使用模型：qwen/qwen3-32b

  Q1：請問蘋果公司(AAPL)目前的股價是多少？
  Q2：台北(Taipei)現在的天氣如何？
  Q3：請給我最新的科技(tech)新聞
  Q4：請問微軟(MSFT)的股價是多少？
  Q5：你好！請問你是什麼人工智慧模型？

✅ 完成，結果已寫入 fc_log.json（共 5 題）


In [17]:
for i, result in enumerate(results):
    print(f"Q{i+1}：{result['query']}")
    print(f"A{i+1}：{result['final_answer']}")
    print("="*30)

Q1：請問蘋果公司(AAPL)目前的股價是多少？
A1：蘋果公司(AAPL)目前的股價為150.25美元，較昨日上漲1.2%。
Q2：台北(Taipei)現在的天氣如何？
A2：目前台北的氣溫為25°C，天氣多雲，濕度70%，風速12公里/小時。
Q3：請給我最新的科技(tech)新聞
A3：以下是近期科技新聞摘要：
1. 2024-01-24 蘋果 Vision Pro 2 發表，重量減至 400g，電池續航提升至 4 小時，售價 3,499 美元。
2. 2024-01-23 微軟 Copilot 整合 Office 全系列，文件處理時間平均縮短 35%。
3. 2024-01-22 Anthropic Claude 4 在程式碼測試得分 92.3，超越 GPT-5 的 89.7 分。
4. 2024-01-21 NVIDIA H200 GPU 等待期超 12 個月，雲端廠商已預訂 2025 全年出貨。
5. 2024-01-20 特斯拉 FSD v13 在美國全面開放，事故率較人類駕駛降低 45%。
6. 2024-01-19 Meta 推出 98g AR 眼鏡 Orion，視野 70 度，2025 年限量上市。
7. 2024-01-18 三星 Galaxy S25 搭載自研 Exynos 2500，相機處理速度提升 60%。
8. 2024-01-17 台積電 2nm 製程良率達 75%，2025 年供應 Apple 與 NVIDIA。
9. 2024-01-16 Google 量子計算實現 1000 位元操作，錯誤率低於 0.1%。
10. 2024-01-15 GPT-5 正式發布，API 定價降 40%，效能提升 3 倍。
Q4：請問微軟(MSFT)的股價是多少？
A4：執行錯誤：Error code: 429 - {'error': {'message': 'Rate limit reached for model `qwen/qwen3-32b` in organization `org_01kr03k1hpfmbrzdh7s2taj3rx` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 5079, Requested 1538. Please try 

## 8. 加分題：跨模型比較（+10）            
跑第二個模型，產出另一份 fc_log，在報告中分析兩個模型的差異（工具選擇、參數抽取、抗幻覺、閒聊判斷）。

In [27]:
# 加分題
representative_models = [
    "llama-3.3-70b-versatile",
    "llama-3.1-8b-instant"
]

for model in representative_models:
    print(f"正在測試模型: {model}...")
    set_model(model)

    filename = f"fc_log_{model.split('/')[-1]}.json"
    run_tests(run_agent, db, output_file=filename)

print("\n✅ 測試完成！")

正在測試模型: llama-3.3-70b-versatile...
使用模型：llama-3.3-70b-versatile

  Q1：請問蘋果公司(AAPL)目前的股價是多少？
  Q2：台北(Taipei)現在的天氣如何？
  Q3：請給我最新的科技(tech)新聞
  Q4：請問微軟(MSFT)的股價是多少？
  Q5：你好！請問你是什麼人工智慧模型？

✅ 完成，結果已寫入 fc_log_llama-3.3-70b-versatile.json（共 5 題）
正在測試模型: llama-3.1-8b-instant...
使用模型：llama-3.1-8b-instant

  Q1：請問蘋果公司(AAPL)目前的股價是多少？
  Q2：台北(Taipei)現在的天氣如何？
  Q3：請給我最新的科技(tech)新聞
  Q4：請問微軟(MSFT)的股價是多少？
  Q5：你好！請問你是什麼人工智慧模型？

✅ 完成，結果已寫入 fc_log_llama-3.1-8b-instant.json（共 5 題）

✅ 測試完成！
